In [1]:
import pandas as pd
import numpy as np
import os
from scipy.interpolate import PchipInterpolator
import warnings
warnings.filterwarnings("ignore") 

# 1: initialize configuration and 15-minute master timeline
START_DATE = '2024-01-01 00:00:00'
END_DATE = '2025-07-01 00:00:00'
MASTER_TIMELINE = pd.date_range(start=START_DATE, end=END_DATE, freq='15min')

print("Starting Phase 1: Data Preparation and Resolution Alignment...")

# 2: define function to align continuous variables using PCHIP or forward fill
def process_variable(df_master, folder, filename, col_name, method):
    file_path = os.path.join(folder, filename + '.xlsx')
    if not os.path.exists(file_path):
        print(f"  [WARNING] Missing file: {file_path}")
        return df_master
    
    print(f"  -> Processing {filename} ({method})...")
    df_var = pd.read_excel(file_path)
    df_var.rename(columns={df_var.columns[0]: 'Time'}, inplace=True)
    df_var['Time'] = pd.to_datetime(df_var['Time'])
    df_var.set_index('Time', inplace=True)
    df_var = df_var[~df_var.index.duplicated(keep='first')]
    
    aligned_series = df_var[col_name].reindex(MASTER_TIMELINE)
    
    if method == 'ffill':
        df_master[col_name] = aligned_series.ffill()
    elif method == 'pchip':
        valid_data = aligned_series.dropna()
        if len(valid_data) > 3:
            x_known = np.arange(len(aligned_series))[aligned_series.notna()]
            y_known = valid_data.values
            interpolator = PchipInterpolator(x_known, y_known)
            x_all = np.arange(len(aligned_series))
            df_master[col_name] = interpolator(x_all)
            
    return df_master

# 3: define function to average and align stochastic rainfall data
def process_rainfall(df_master, folder, filename):
    file_path = os.path.join(folder, filename + '.xlsx')
    if not os.path.exists(file_path):
        print(f"  [WARNING] Missing file: {file_path}")
        return df_master
        
    print(f"  -> Processing {filename} (Averaging Stations -> ffill)...")
    df_rain = pd.read_excel(file_path)
    df_rain.rename(columns={df_rain.columns[0]: 'Time'}, inplace=True)
    df_rain['Time'] = pd.to_datetime(df_rain['Time'])
    df_rain.set_index('Time', inplace=True)
    df_rain = df_rain[~df_rain.index.duplicated(keep='first')]
    
    # Isolate station columns and calculate the daily mean
    station_cols = [c for c in df_rain.columns if str(c).startswith('S')]
    df_rain['Rainfall'] = df_rain[station_cols].mean(axis=1)
    
    aligned_rain = df_rain['Rainfall'].reindex(MASTER_TIMELINE)
    df_master['Rainfall'] = aligned_rain.ffill()
    
    return df_master

# 4: define function to aggregate high-frequency power generation data
def process_rtd(df_master, folder, filename, units):
    file_path = os.path.join(folder, filename + '.xlsx')
    if not os.path.exists(file_path):
        print(f"  [WARNING] Missing file: {file_path}")
        return df_master
        
    print(f"  -> Processing {filename} (Summing Units -> Mean Aggregation)...")
    df_rtd = pd.read_excel(file_path)
    df_rtd.rename(columns={df_rtd.columns[0]: 'Time'}, inplace=True)
    df_rtd['Time'] = pd.to_datetime(df_rtd['Time'])
    df_rtd.set_index('Time', inplace=True)
    df_rtd = df_rtd[~df_rtd.index.duplicated(keep='first')]
    
    # Calculate total power
    df_rtd['Total_Power'] = df_rtd[units].sum(axis=1)
    
    # Downsample to 15-min
    df_power_15m = df_rtd[['Total_Power']].resample('15min').mean()
    
    return df_master.join(df_power_15m)

# 5: process and export multirate data for Agus I
df_agus1 = pd.DataFrame(index=MASTER_TIMELINE)
df_agus1.index.name = 'Time'

df_agus1 = process_rtd(df_agus1, 'Agus1', '1_Agus1_RTD', ['U1', 'U2'])
df_agus1 = process_variable(df_agus1, 'Agus1', '2_Agus1_LakeElevation', 'Elevation', 'pchip')
df_agus1 = process_variable(df_agus1, 'Agus1', '3_Agus1_CMS', 'CMS', 'pchip')
df_agus1 = process_variable(df_agus1, 'Agus1', '4_Agus1_Outflow', 'Outflow', 'pchip')
df_agus1 = process_variable(df_agus1, 'Agus1', '5_Agus1_LakeInflow', 'Lake_Inflow', 'pchip')
df_agus1 = process_variable(df_agus1, 'Agus1', '6_Agus1_LakeOutflow', 'Lake_Outflow', 'pchip')
df_agus1 = process_variable(df_agus1, 'Agus1', '7_Agus1_Spillage', 'Spillage', 'pchip')
df_agus1 = process_rainfall(df_agus1, 'Agus1', '8_Agus1_Rainfall')

df_agus1 = df_agus1.bfill().ffill() # Clean up any trailing NaNs
df_agus1.to_csv('Processed_Agus1_Dataset.csv')
print(f"SUCCESS: Saved Processed_Agus1_Dataset.csv | Shape: {df_agus1.shape}")

# 6: process and export multirate data for Agus I
df_agus2 = pd.DataFrame(index=MASTER_TIMELINE)
df_agus2.index.name = 'Time'

df_agus2 = process_rtd(df_agus2, 'Agus2', '1_Agus2_RTD', ['U1', 'U2', 'U3'])
df_agus2 = process_variable(df_agus2, 'Agus2', '2_Agus2_Forebay', 'Forebay', 'pchip')
df_agus2 = process_variable(df_agus2, 'Agus2', '3_Agus2_Inflow', 'Inflow', 'pchip') 
df_agus2 = process_variable(df_agus2, 'Agus2', '4_Agus2_Outflow', 'Outflow', 'pchip')
df_agus2 = process_variable(df_agus2, 'Agus2', '5_Agus2_Spillage', 'Spillage', 'pchip')
df_agus2 = process_rainfall(df_agus2, 'Agus2', '6_Agus2_Rainfall')

df_agus2 = df_agus2.bfill().ffill()
df_agus2.to_csv('Processed_Agus2_Dataset.csv')

Starting Phase 1: Data Preparation and Resolution Alignment...

--- PROCESSING AGUS 1 ---
  -> Processing 1_Agus1_RTD (Summing Units -> Mean Aggregation)...
  -> Processing 2_Agus1_LakeElevation (pchip)...
  -> Processing 3_Agus1_CMS (pchip)...
  -> Processing 4_Agus1_Outflow (pchip)...
  -> Processing 5_Agus1_LakeInflow (pchip)...
  -> Processing 6_Agus1_LakeOutflow (pchip)...
  -> Processing 7_Agus1_Spillage (pchip)...
  -> Processing 8_Agus1_Rainfall (Averaging Stations -> ffill)...
SUCCESS: Saved Processed_Agus1_Dataset.csv | Shape: (52513, 8)

--- PROCESSING AGUS 2 ---
  -> Processing 1_Agus2_RTD (Summing Units -> Mean Aggregation)...
  -> Processing 2_Agus2_Forebay (pchip)...
  -> Processing 3_Agus2_Inflow (pchip)...
  -> Processing 4_Agus2_Outflow (pchip)...
  -> Processing 5_Agus2_Spillage (pchip)...
  -> Processing 6_Agus2_Rainfall (Averaging Stations -> ffill)...
SUCCESS: Saved Processed_Agus2_Dataset.csv | Shape: (52513, 6)

*** ALL DATA PROCESSING COMPLETE! ***
